# Week 10: Hyperparameter Tuning & Learning Curves
# 第 10 週：超參數調校與學習曲線

## 學習目標 Learning Objectives
1. 理解超參數 vs 模型參數的區別
2. 使用 GridSearchCV 進行窮舉搜尋
3. 使用 RandomizedSearchCV 進行隨機搜尋，比較與 Grid Search 的差異
4. 透過學習曲線 (Learning Curve) 診斷過擬合 / 欠擬合
5. 透過驗證曲線 (Validation Curve) 選擇最佳超參數
6. 比較不同交叉驗證策略
7. 使用巢狀交叉驗證 (Nested CV) 進行無偏模型評估
8. 建構完整的超參數調校流程

In [ ]:
# 匯入必要套件 Import required packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.colors import ListedColormap

from sklearn.datasets import load_iris, load_wine
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    GridSearchCV, RandomizedSearchCV,
    learning_curve, validation_curve,
    cross_val_score, KFold, StratifiedKFold, RepeatedStratifiedKFold,
    train_test_split
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, make_scorer
from scipy.stats import uniform, randint
import warnings
warnings.filterwarnings('ignore')

# 設定中文字型
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

np.random.seed(42)

# 載入資料集
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

wine = load_wine()
X_wine, y_wine = wine.data, wine.target

print(f'Iris dataset: {X_iris.shape[0]} samples, {X_iris.shape[1]} features, {len(np.unique(y_iris))} classes')
print(f'Wine dataset: {X_wine.shape[0]} samples, {X_wine.shape[1]} features, {len(np.unique(y_wine))} classes')
print('\nAll packages imported successfully!')

---
## Part 1: 超參數 vs 模型參數 Hyperparameters vs Model Parameters

在機器學習中，有兩類截然不同的「參數」：

| 比較項目 | 模型參數 Model Parameters | 超參數 Hyperparameters |
|----------|:---:|:---:|
| 定義 | 模型從資料中自動學習的數值 | 訓練前由人工設定的配置值 |
| 學習方式 | 由訓練演算法最佳化 | 由人工或搜尋演算法決定 |
| 範例 | 權重 $w$、偏差 $b$ | 學習率、正則化強度 $C$、樹深度 |

> **類比：** 模型參數就像考試時學生填寫的答案，超參數則像老師設計的考試規則（考試時長、題目數量等）。

In [ ]:
# 展示超參數對 SVM 的影響
# SVM 的超參數 C 控制正則化強度

X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 嘗試不同的 C 值
C_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
train_accs = []
test_accs = []

for C in C_values:
    svm = SVC(C=C, kernel='rbf', random_state=42)
    svm.fit(X_train_s, y_train)
    train_accs.append(svm.score(X_train_s, y_train))
    test_accs.append(svm.score(X_test_s, y_test))

# 繪圖
fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogx(C_values, train_accs, 'bo-', linewidth=2, markersize=8, label='Train Accuracy')
ax.semilogx(C_values, test_accs, 'rs-', linewidth=2, markersize=8, label='Test Accuracy')
ax.set_xlabel('C (Regularization Parameter)', fontsize=13)
ax.set_ylabel('Accuracy', fontsize=13)
ax.set_title('SVM: Hyperparameter C vs Accuracy (Iris)', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_ylim([0.5, 1.05])

# 標註最佳
best_idx = np.argmax(test_accs)
ax.annotate(f'Best C={C_values[best_idx]}\nTest Acc={test_accs[best_idx]:.3f}',
            xy=(C_values[best_idx], test_accs[best_idx]),
            xytext=(C_values[best_idx]*5, test_accs[best_idx]-0.08),
            fontsize=11, arrowprops=dict(arrowstyle='->', color='red'),
            color='red', fontweight='bold')
plt.tight_layout()
plt.show()

print('Key Observations:')
print('1. C too small -> strong regularization -> underfitting (both train & test low)')
print('2. C too large -> weak regularization -> potential overfitting')
print('3. We need to find a sweet spot -- this is hyperparameter tuning!')

In [ ]:
# 模型參數 vs 超參數的直觀對比
print('=== SVM (C=1, RBF kernel) Parameter Types ===')
svm_demo = SVC(C=1, kernel='rbf', gamma='scale', random_state=42)
svm_demo.fit(X_train_s, y_train)

print(f'\nHyperparameters (set before training):')
print(f'  C = {svm_demo.C}')
print(f'  kernel = {svm_demo.kernel}')
print(f'  gamma = {svm_demo.gamma}')

print(f'\nModel Parameters (learned during training):')
print(f'  n_support_vectors = {svm_demo.n_support_}')
print(f'  support_vectors shape = {svm_demo.support_vectors_.shape}')
print(f'  intercept = {svm_demo.intercept_}')

print('\nCommon model hyperparameters:')
print('  SVM         -> C, kernel, gamma')
print('  RandomForest -> n_estimators, max_depth, min_samples_split')
print('  KNN         -> n_neighbors, metric, weights')
print('  Neural Net  -> learning_rate, hidden_layer_sizes, alpha')

---
## Part 2: GridSearchCV — 窮舉搜尋 Exhaustive Grid Search

Grid Search 對每個超參數定義一組候選值，然後**窮舉所有可能的組合**，逐一用交叉驗證評估模型效能。

$$\text{Total evaluations} = |\text{param}_1| \times |\text{param}_2| \times \cdots \times k_{\text{folds}}$$

In [ ]:
# GridSearchCV for RandomForest on Wine dataset
X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_wine, y_wine, test_size=0.3, random_state=42, stratify=y_wine
)

param_grid = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [3, 5, 7, 10, None],
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    return_train_score=True
)
grid_search.fit(X_train_w, y_train_w)

print(f'Search space size: {len(param_grid["n_estimators"])} x {len(param_grid["max_depth"])} = '
      f'{len(param_grid["n_estimators"]) * len(param_grid["max_depth"])} combinations')
print(f'With 5-fold CV -> {len(param_grid["n_estimators"]) * len(param_grid["max_depth"]) * 5} total fits')
print(f'\nBest Params: {grid_search.best_params_}')
print(f'Best CV Score: {grid_search.best_score_:.4f}')
print(f'Test Score: {grid_search.score(X_test_w, y_test_w):.4f}')

In [ ]:
# 繪製 GridSearchCV 結果熱力圖
results_df = grid_search.cv_results_

n_est_vals = param_grid['n_estimators']
depth_vals = param_grid['max_depth']
depth_labels = [str(d) for d in depth_vals]
score_matrix = np.zeros((len(depth_vals), len(n_est_vals)))

for i, depth in enumerate(depth_vals):
    for j, n_est in enumerate(n_est_vals):
        mask = []
        for p in results_df['params']:
            mask.append(p['n_estimators'] == n_est and p['max_depth'] == depth)
        mask = np.array(mask)
        if mask.any():
            score_matrix[i, j] = results_df['mean_test_score'][mask][0]

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(score_matrix, cmap='YlOrRd', aspect='auto',
               vmin=score_matrix.min() - 0.01)

for i in range(len(depth_labels)):
    for j in range(len(n_est_vals)):
        color = 'white' if score_matrix[i, j] > score_matrix.mean() + 0.01 else 'black'
        ax.text(j, i, f'{score_matrix[i, j]:.3f}', ha='center', va='center',
                fontsize=12, fontweight='bold', color=color)

ax.set_xticks(range(len(n_est_vals)))
ax.set_xticklabels(n_est_vals, fontsize=11)
ax.set_yticks(range(len(depth_labels)))
ax.set_yticklabels(depth_labels, fontsize=11)
ax.set_xlabel('n_estimators', fontsize=13)
ax.set_ylabel('max_depth', fontsize=13)
ax.set_title('GridSearchCV Heatmap: RandomForest on Wine', fontsize=14)
plt.colorbar(im, ax=ax, label='Mean CV Accuracy')
plt.tight_layout()
plt.show()

print(f'Best combination: n_estimators={grid_search.best_params_["n_estimators"]}, '
      f'max_depth={grid_search.best_params_["max_depth"]}')
print('The brightest cell in the heatmap = best CV accuracy')

---
## Part 3: RandomizedSearchCV — 隨機搜尋 Random Search

Random Search 從超參數空間中**隨機採樣**固定次數的組合，支援從連續分布中取樣。

Bergstra & Bengio (2012) 證明：**當只有少數超參數真正重要時，Random Search 比 Grid Search 更高效。**

In [ ]:
# RandomizedSearchCV with continuous distributions
param_distributions = {
    'n_estimators': randint(50, 500),
    'max_depth': randint(2, 20),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_distributions,
    n_iter=20,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    return_train_score=True
)
random_search.fit(X_train_w, y_train_w)

print('RandomizedSearchCV Results:')
print(f'  n_iter = 20')
print(f'  Best Params: {random_search.best_params_}')
print(f'  Best CV Score: {random_search.best_score_:.4f}')
print(f'  Test Score: {random_search.score(X_test_w, y_test_w):.4f}')
print(f'\nComparison with GridSearchCV:')
print(f'  GridSearchCV best score:   {grid_search.best_score_:.4f} (20 combos, 100 fits)')
print(f'  RandomSearchCV best score: {random_search.best_score_:.4f} (20 combos, 100 fits)')

In [ ]:
# 比較 Grid Search vs Random Search 的搜尋效率
np.random.seed(42)

scaler_iris = StandardScaler()
X_iris_s = scaler_iris.fit_transform(X_iris)

def evaluate_svm(C, gamma, X, y, cv=3):
    """Quick evaluation of SVM with given hyperparams."""
    svm = SVC(C=C, gamma=gamma, kernel='rbf', random_state=42)
    scores = cross_val_score(svm, X, y, cv=cv, scoring='accuracy')
    return scores.mean()

# Grid Search: fixed grid
C_grid = np.logspace(-2, 3, 8)
gamma_grid = np.logspace(-4, 1, 8)

grid_best_so_far = []
current_best = 0
for C in C_grid:
    for gamma in gamma_grid:
        score = evaluate_svm(C, gamma, X_iris_s, y_iris)
        current_best = max(current_best, score)
        grid_best_so_far.append(current_best)

# Random Search: random sampling
n_random = len(grid_best_so_far)
random_best_so_far = []
current_best = 0
for i in range(n_random):
    C = 10 ** np.random.uniform(-2, 3)
    gamma = 10 ** np.random.uniform(-4, 1)
    score = evaluate_svm(C, gamma, X_iris_s, y_iris)
    current_best = max(current_best, score)
    random_best_so_far.append(current_best)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: convergence
ax1 = axes[0]
ax1.plot(range(1, len(grid_best_so_far)+1), grid_best_so_far, 'b-', linewidth=2,
         label=f'Grid Search (final: {grid_best_so_far[-1]:.4f})')
ax1.plot(range(1, len(random_best_so_far)+1), random_best_so_far, 'r-', linewidth=2,
         label=f'Random Search (final: {random_best_so_far[-1]:.4f})')
ax1.set_xlabel('Number of Evaluations', fontsize=12)
ax1.set_ylabel('Best Score So Far', fontsize=12)
ax1.set_title('Convergence: Grid vs Random Search', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Right: search point distribution
ax2 = axes[1]
C_g, G_g = np.meshgrid(C_grid, gamma_grid)
ax2.scatter(C_g.ravel(), G_g.ravel(), c='blue', s=50, alpha=0.6,
            label='Grid Search', marker='s')
C_r = 10 ** np.random.uniform(-2, 3, n_random)
G_r = 10 ** np.random.uniform(-4, 1, n_random)
ax2.scatter(C_r, G_r, c='red', s=50, alpha=0.6, label='Random Search', marker='o')
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('C', fontsize=12)
ax2.set_ylabel('gamma', fontsize=12)
ax2.set_title('Search Space Coverage', fontsize=14)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Key observations:')
print('1. Random Search often converges faster to a good solution')
print('2. Grid points form a regular lattice -> wastes computation on unimportant dimensions')
print('3. Random points spread more widely -> explores more unique values per dimension')

---
## Part 4: 學習曲線 Learning Curves

學習曲線描繪**隨著訓練資料量增加**，模型在訓練集和驗證集上的表現如何變化。
它是診斷過擬合 (Overfitting) 與欠擬合 (Underfitting) 的強大工具。

$$\text{Learning Curve: } \text{Score} = f(\text{Training Set Size})$$

In [ ]:
# 繪製三種典型學習曲線
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = {
    'High Bias (Underfitting)\nDecisionTree(max_depth=1)': DecisionTreeClassifier(
        max_depth=1, random_state=42),
    'Good Fit\nSVM(C=1, RBF)': SVC(C=1, kernel='rbf', random_state=42),
    'High Variance (Overfitting)\nDecisionTree(max_depth=None)': DecisionTreeClassifier(
        max_depth=None, random_state=42),
}

scaler_lc = StandardScaler()
X_wine_s = scaler_lc.fit_transform(X_wine)

for idx, (title, model) in enumerate(models.items()):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_wine_s, y_wine,
        train_sizes=np.linspace(0.1, 1.0, 10),
        cv=5, scoring='accuracy', n_jobs=-1, random_state=42
    )
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)

    ax = axes[idx]
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                    alpha=0.15, color='blue')
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                    alpha=0.15, color='red')
    ax.plot(train_sizes, train_mean, 'bo-', linewidth=2, markersize=5,
            label='Training Score')
    ax.plot(train_sizes, val_mean, 'rs-', linewidth=2, markersize=5,
            label='Validation Score')
    ax.set_xlabel('Training Set Size', fontsize=11)
    ax.set_ylabel('Accuracy', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(loc='lower right', fontsize=9)
    ax.set_ylim([0.3, 1.05])
    ax.grid(True, alpha=0.3)

plt.suptitle('Learning Curves: Three Typical Patterns', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print('Three typical patterns:')
print('1. High Bias (Underfitting): both train & val low and close -> increase model complexity')
print('2. Good Fit: both train & val high with small gap -> model is appropriate')
print('3. High Variance (Overfitting): train high but val low -> get more data or regularize')

---
## Part 5: 驗證曲線 Validation Curves

驗證曲線描繪**隨著某個超參數值的變化**，模型在訓練集和驗證集上的表現。
與學習曲線不同，x 軸是超參數值而非資料量。

$$\text{Validation Curve: } \text{Score} = f(\text{Hyperparameter Value})$$

In [ ]:
# 驗證曲線：SVM gamma 與 RandomForest max_depth
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: SVM gamma
gamma_range = np.logspace(-6, 2, 20)
train_scores_g, val_scores_g = validation_curve(
    SVC(C=1, kernel='rbf', random_state=42),
    X_wine_s, y_wine,
    param_name='gamma', param_range=gamma_range,
    cv=5, scoring='accuracy', n_jobs=-1
)
train_mean_g = train_scores_g.mean(axis=1)
train_std_g = train_scores_g.std(axis=1)
val_mean_g = val_scores_g.mean(axis=1)
val_std_g = val_scores_g.std(axis=1)

ax1 = axes[0]
ax1.fill_between(gamma_range, train_mean_g - train_std_g, train_mean_g + train_std_g,
                 alpha=0.15, color='blue')
ax1.fill_between(gamma_range, val_mean_g - val_std_g, val_mean_g + val_std_g,
                 alpha=0.15, color='red')
ax1.semilogx(gamma_range, train_mean_g, 'bo-', linewidth=2, markersize=5,
             label='Training Score')
ax1.semilogx(gamma_range, val_mean_g, 'rs-', linewidth=2, markersize=5,
             label='Validation Score')
best_g = np.argmax(val_mean_g)
ax1.axvline(x=gamma_range[best_g], color='green', linestyle='--', linewidth=2,
            label=f'Best gamma={gamma_range[best_g]:.4f}')
ax1.set_xlabel('gamma', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('Validation Curve: SVM gamma (Wine)', fontsize=14)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0.3, 1.05])

# Right: RandomForest max_depth
depth_range = np.arange(1, 25)
train_scores_d, val_scores_d = validation_curve(
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_wine, y_wine,
    param_name='max_depth', param_range=depth_range,
    cv=5, scoring='accuracy', n_jobs=-1
)
train_mean_d = train_scores_d.mean(axis=1)
train_std_d = train_scores_d.std(axis=1)
val_mean_d = val_scores_d.mean(axis=1)
val_std_d = val_scores_d.std(axis=1)

ax2 = axes[1]
ax2.fill_between(depth_range, train_mean_d - train_std_d, train_mean_d + train_std_d,
                 alpha=0.15, color='blue')
ax2.fill_between(depth_range, val_mean_d - val_std_d, val_mean_d + val_std_d,
                 alpha=0.15, color='red')
ax2.plot(depth_range, train_mean_d, 'bo-', linewidth=2, markersize=5,
         label='Training Score')
ax2.plot(depth_range, val_mean_d, 'rs-', linewidth=2, markersize=5,
         label='Validation Score')
best_d = np.argmax(val_mean_d)
ax2.axvline(x=depth_range[best_d], color='green', linestyle='--', linewidth=2,
            label=f'Best max_depth={depth_range[best_d]}')
ax2.set_xlabel('max_depth', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Validation Curve: RandomForest max_depth (Wine)', fontsize=14)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0.7, 1.05])

plt.tight_layout()
plt.show()

print('Reading validation curves:')
print('  Left side: param too small -> underfitting (both scores low)')
print('  Middle: validation score peaks = best hyperparameter value')
print('  Right side: param too large -> overfitting (train high, val drops)')

---
## Part 6: 交叉驗證策略比較 Cross-Validation Strategies

不同的交叉驗證策略在資料分割方式上有所不同：

| 策略 | 說明 |
|------|------|
| KFold | 隨機分割，不保證類別比例 |
| StratifiedKFold | 保持每折中的類別比例一致 |
| RepeatedStratifiedKFold | 重複多次 StratifiedKFold，更穩定的估計 |

In [ ]:
# 比較不同 CV 策略
np.random.seed(42)

model_cv = SVC(C=1, kernel='rbf', random_state=42)

cv_strategies = {
    'KFold (k=5)': KFold(n_splits=5, shuffle=True, random_state=42),
    'StratifiedKFold (k=5)': StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'RepeatedStratifiedKFold\n(k=5, n=3)': RepeatedStratifiedKFold(
        n_splits=5, n_repeats=3, random_state=42)
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
all_results = {}

for idx, (name, cv) in enumerate(cv_strategies.items()):
    scores = cross_val_score(model_cv, X_wine_s, y_wine, cv=cv, scoring='accuracy')
    all_results[name] = scores

    ax = axes[idx]
    colors = plt.cm.Set2(np.linspace(0, 1, len(scores)))
    ax.bar(range(len(scores)), scores, color=colors, edgecolor='black', alpha=0.8)
    ax.axhline(y=scores.mean(), color='red', linestyle='--', linewidth=2,
               label=f'Mean = {scores.mean():.4f}')
    ax.fill_between(range(len(scores)),
                    scores.mean() - scores.std(), scores.mean() + scores.std(),
                    alpha=0.1, color='red')
    ax.set_xlabel('Fold', fontsize=11)
    ax.set_ylabel('Accuracy', fontsize=11)
    ax.set_title(f'{name}\nStd = {scores.std():.4f}', fontsize=12)
    ax.legend(fontsize=10)
    ax.set_ylim([0.8, 1.05])
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Cross-Validation Strategy Comparison (SVM on Wine)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('\n=== CV Strategy Comparison ===')
for name, scores in all_results.items():
    clean_name = name.replace('\n', ' ')
    print(f'{clean_name:45s}: Mean={scores.mean():.4f}, Std={scores.std():.4f}, '
          f'Folds={len(scores)}')
print('\nRecommendations:')
print('- Small datasets: use RepeatedStratifiedKFold for more stable estimates')
print('- General case: use StratifiedKFold to preserve class ratios')
print('- KFold may produce biased estimates on imbalanced datasets')

In [ ]:
# 視覺化不同 CV 策略如何分割資料
fig, axes = plt.subplots(3, 1, figsize=(14, 8))

cv_demos = [
    ('KFold (k=5)', KFold(n_splits=5, shuffle=True, random_state=42)),
    ('StratifiedKFold (k=5)', StratifiedKFold(n_splits=5, shuffle=True, random_state=42)),
    ('RepeatedStratifiedKFold (k=5, n=2)', RepeatedStratifiedKFold(
        n_splits=5, n_repeats=2, random_state=42))
]

colors_class = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for ax_idx, (name, cv) in enumerate(cv_demos):
    ax = axes[ax_idx]
    folds = list(cv.split(X_wine, y_wine))
    n_folds = min(len(folds), 10)

    for fold_idx in range(n_folds):
        train_idx, val_idx = folds[fold_idx]
        for i in range(len(y_wine)):
            if i in val_idx:
                color = colors_class[y_wine[i]]
                ax.scatter(i, fold_idx, c=color, s=3, marker='s', alpha=0.9)
            else:
                ax.scatter(i, fold_idx, c='lightgray', s=1, marker='s', alpha=0.3)

    ax.set_ylabel('Fold #', fontsize=11)
    ax.set_title(name, fontsize=12)
    ax.set_yticks(range(n_folds))
    ax.set_xlim([-1, len(y_wine)])

axes[-1].set_xlabel('Sample Index', fontsize=11)
plt.suptitle('CV Data Splitting (colored = validation set, by class)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('Colored blocks = validation samples (color = class label)')
print('Gray = training samples')
print('StratifiedKFold ensures each fold has consistent class proportions')

---
## Part 7: 巢狀交叉驗證 Nested Cross-Validation

巢狀交叉驗證是進行**無偏模型評估**的正確方式：
- **外層 CV (Outer CV)**：評估模型的泛化能力
- **內層 CV (Inner CV)**：用於超參數調校

如果用同一個 CV 既做調參又做評估，會產生**樂觀偏差 (Optimistic Bias)**。

In [ ]:
# 巢狀 CV vs 非巢狀 CV 的比較
np.random.seed(42)

param_grid_nested = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1]
}

# Method 1: Non-nested CV (biased)
gs_non_nested = GridSearchCV(
    SVC(kernel='rbf', random_state=42),
    param_grid_nested, cv=5, scoring='accuracy', n_jobs=-1
)
gs_non_nested.fit(X_wine_s, y_wine)
non_nested_score = gs_non_nested.best_score_

# Method 2: Nested CV (unbiased)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
nested_scores = []

for train_idx, test_idx in outer_cv.split(X_wine_s, y_wine):
    X_tr, X_te = X_wine_s[train_idx], X_wine_s[test_idx]
    y_tr, y_te = y_wine[train_idx], y_wine[test_idx]

    inner_gs = GridSearchCV(
        SVC(kernel='rbf', random_state=42),
        param_grid_nested, cv=3, scoring='accuracy', n_jobs=-1
    )
    inner_gs.fit(X_tr, y_tr)
    nested_scores.append(inner_gs.score(X_te, y_te))

nested_scores = np.array(nested_scores)

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(['Non-Nested CV\n(Biased)', 'Nested CV\n(Unbiased)'],
              [non_nested_score, nested_scores.mean()],
              color=['#FF6B6B', '#4ECDC4'], edgecolor='black',
              yerr=[0, nested_scores.std()], capsize=10, alpha=0.8)
ax.set_ylabel('Accuracy', fontsize=13)
ax.set_title('Nested vs Non-Nested Cross-Validation', fontsize=14)
ax.set_ylim([0.9, 1.01])
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, [non_nested_score, nested_scores.mean()]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Non-nested CV score (optimistic bias): {non_nested_score:.4f}')
print(f'Nested CV score (unbiased estimate):   {nested_scores.mean():.4f} +/- {nested_scores.std():.4f}')
print(f'Bias difference: {non_nested_score - nested_scores.mean():.4f}')
print('\nNested CV estimate is usually lower because it is not contaminated by the tuning process.')

---
## Part 8: 實務超參數調校流程 Practical Tuning Workflow

完整的超參數調校 Pipeline 建議：
1. 先用 RandomizedSearchCV 粗搜尋找出有潛力的區域
2. 再用 GridSearchCV 細搜尋精確定位最佳值
3. 使用 Nested CV 確認最終效能的無偏估計

In [ ]:
# 完整調校流程示範
np.random.seed(42)

print('=' * 60)
print('  Complete Hyperparameter Tuning Pipeline')
print('=' * 60)

# Step 1: data preparation
X_tr_final, X_te_final, y_tr_final, y_te_final = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42, stratify=y_wine
)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', random_state=42))
])

print('\n--- Step 1: Coarse Random Search ---')
coarse_params = {
    'svm__C': uniform(0.01, 100),
    'svm__gamma': uniform(0.0001, 1)
}
coarse_search = RandomizedSearchCV(
    pipe, coarse_params, n_iter=30, cv=5,
    scoring='accuracy', random_state=42, n_jobs=-1
)
coarse_search.fit(X_tr_final, y_tr_final)

best_C_coarse = coarse_search.best_params_['svm__C']
best_gamma_coarse = coarse_search.best_params_['svm__gamma']
print(f'Coarse best: C={best_C_coarse:.4f}, gamma={best_gamma_coarse:.6f}')
print(f'Coarse best CV score: {coarse_search.best_score_:.4f}')

# Step 2: Fine Grid Search
print('\n--- Step 2: Fine Grid Search ---')
fine_C = np.linspace(max(0.01, best_C_coarse * 0.5), best_C_coarse * 2, 5)
fine_gamma = np.linspace(max(0.0001, best_gamma_coarse * 0.5), best_gamma_coarse * 2, 5)

fine_params = {'svm__C': fine_C, 'svm__gamma': fine_gamma}
fine_search = GridSearchCV(
    pipe, fine_params, cv=5, scoring='accuracy', n_jobs=-1
)
fine_search.fit(X_tr_final, y_tr_final)

print(f'Fine best: C={fine_search.best_params_["svm__C"]:.4f}, '
      f'gamma={fine_search.best_params_["svm__gamma"]:.6f}')
print(f'Fine best CV score: {fine_search.best_score_:.4f}')

# Step 3: Final evaluation
print('\n--- Step 3: Final Evaluation ---')
final_score = fine_search.score(X_te_final, y_te_final)
print(f'Final Test Score: {final_score:.4f}')
print(f'Best Parameters: {fine_search.best_params_}')

In [ ]:
# 視覺化調校流程的搜尋軌跡
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Coarse Search results
ax1 = axes[0]
coarse_results = coarse_search.cv_results_
C_vals = [p['svm__C'] for p in coarse_results['params']]
gamma_vals = [p['svm__gamma'] for p in coarse_results['params']]
mean_scores = coarse_results['mean_test_score']

sc1 = ax1.scatter(C_vals, gamma_vals, c=mean_scores, cmap='viridis',
                  s=100, edgecolors='black', alpha=0.8, vmin=0.85)
ax1.scatter([best_C_coarse], [best_gamma_coarse], c='red', s=300,
            marker='*', edgecolors='black', linewidths=2, zorder=5, label='Best')
ax1.set_xlabel('C', fontsize=12)
ax1.set_ylabel('gamma', fontsize=12)
ax1.set_title('Step 1: Coarse Random Search', fontsize=13)
ax1.legend(fontsize=12)
plt.colorbar(sc1, ax=ax1, label='CV Accuracy')

# Right: Fine Search results
ax2 = axes[1]
fine_results = fine_search.cv_results_
C_fine_vals = [p['svm__C'] for p in fine_results['params']]
gamma_fine_vals = [p['svm__gamma'] for p in fine_results['params']]
mean_scores_fine = fine_results['mean_test_score']

sc2 = ax2.scatter(C_fine_vals, gamma_fine_vals, c=mean_scores_fine, cmap='viridis',
                  s=100, edgecolors='black', alpha=0.8, vmin=0.85)
ax2.scatter([fine_search.best_params_['svm__C']],
            [fine_search.best_params_['svm__gamma']],
            c='red', s=300, marker='*', edgecolors='black', linewidths=2,
            zorder=5, label='Best')
ax2.set_xlabel('C', fontsize=12)
ax2.set_ylabel('gamma', fontsize=12)
ax2.set_title('Step 2: Fine Grid Search', fontsize=13)
ax2.legend(fontsize=12)
plt.colorbar(sc2, ax=ax2, label='CV Accuracy')

plt.suptitle('Coarse-to-Fine Search Strategy', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print('Coarse-to-Fine strategy:')
print('1. First use Random Search to broadly explore the space')
print('2. Then use Grid Search to precisely locate the optimum')
print('3. More efficient than exhaustive Grid Search over a large range')

---
## Part 9: 練習題 Exercises

完成以下練習來鞏固本週所學。

### Exercise 1: Tune an SVM with RBF kernel (C and gamma)

使用 GridSearchCV 對 Iris 資料集的 SVM(RBF) 進行超參數調校。

搜尋空間：
- `C`: [0.01, 0.1, 1, 10, 100]
- `gamma`: [0.001, 0.01, 0.1, 1, 10]

繪製結果的熱力圖，並報告最佳參數與分數。

In [ ]:
# Exercise 1: Tune SVM with RBF kernel
# TODO: Use GridSearchCV to tune C and gamma
# TODO: Plot a heatmap of CV scores for each (C, gamma) combination

# scaler_ex = StandardScaler()
# X_iris_ex = scaler_ex.fit_transform(X_iris)
# param_grid_ex = {'C': [0.01, 0.1, 1, 10, 100], 'gamma': [0.001, 0.01, 0.1, 1, 10]}
# gs_ex = GridSearchCV(SVC(kernel='rbf'), param_grid_ex, cv=5, return_train_score=True)
# gs_ex.fit(X_iris_ex, y_iris)
# print(f'Best Params: {gs_ex.best_params_}')
# print(f'Best Score: {gs_ex.best_score_:.4f}')
# # Hint: build a 2D score_matrix and use plt.imshow() for the heatmap

### Exercise 2: Grid vs Random search space coverage

在 2D 空間 (C, gamma) 中，分別產生 25 個 Grid Search 點和 25 個 Random Search 點，
繪製在同一張圖上比較分布差異。

提示：Grid Search 用 5x5 等分點；Random Search 從 log-uniform 分布取樣。

In [ ]:
# Exercise 2: Compare search space coverage
# TODO: Generate 25 Grid Search points and 25 Random Search points
# TODO: Plot them on the same figure with log scale axes

# C_grid_ex = np.logspace(-1, 3, 5)
# gamma_grid_ex = np.logspace(-3, 1, 5)
# C_g, G_g = np.meshgrid(C_grid_ex, gamma_grid_ex)
# plt.scatter(C_g.ravel(), G_g.ravel(), label='Grid', marker='s')
#
# C_random_ex = 10 ** np.random.uniform(-1, 3, 25)
# gamma_random_ex = 10 ** np.random.uniform(-3, 1, 25)
# plt.scatter(C_random_ex, gamma_random_ex, label='Random', marker='o')
# plt.xscale('log'); plt.yscale('log')
# plt.legend(); plt.show()

### Exercise 3: Use learning curves to diagnose if more data would help

對 Wine 資料集，分別繪製以下三個模型的學習曲線：
1. `LogisticRegression(C=0.001)` -- 強正則化
2. `RandomForestClassifier(n_estimators=100, max_depth=5)`
3. `KNeighborsClassifier(n_neighbors=1)` -- 最近鄰

根據學習曲線判斷：哪個模型可能受益於更多資料？哪個模型過擬合？

In [ ]:
# Exercise 3: Learning curves for model diagnosis
# TODO: Plot learning curves for the three models
# TODO: Analyze which model overfits, underfits, or could benefit from more data

# models_ex = [
#     ('Logistic (C=0.001)', LogisticRegression(C=0.001, max_iter=1000)),
#     ('RF (depth=5)', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)),
#     ('KNN (k=1)', KNeighborsClassifier(n_neighbors=1))
# ]
# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# for idx, (name, model) in enumerate(models_ex):
#     train_sizes, train_scores, val_scores = learning_curve(
#         model, X_wine_s, y_wine, cv=5,
#         train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
#     )
#     # Plot train and val curves with fill_between for std
#     # ...

---
## 本週實作總結 Lab Summary

在這次實作中，我們完成了：

1. **超參數 vs 模型參數**：理解兩者的區別與重要性
2. **GridSearchCV**：窮舉搜尋所有參數組合，繪製熱力圖
3. **RandomizedSearchCV**：從連續分布隨機取樣，更高效的探索
4. **學習曲線**：診斷過擬合 / 欠擬合，判斷是否需要更多資料
5. **驗證曲線**：觀察超參數值對訓練/驗證分數的影響
6. **交叉驗證策略**：KFold vs StratifiedKFold vs RepeatedStratifiedKFold
7. **巢狀交叉驗證**：避免樂觀偏差的正確評估方式
8. **實務調校流程**：Coarse-to-Fine 先粗後細策略

### 關鍵收穫 Key Takeaways
- Grid Search 保證找到網格中最佳組合，但計算量隨維度指數增長
- Random Search 在高維空間中更高效，且支援連續分布
- 學習曲線幫助判斷「增加資料」vs「增加模型複雜度」
- 驗證曲線幫助選擇單一超參數的最佳值
- **永遠使用巢狀 CV** 進行最終的模型效能評估

### 下週預告 Next Week Preview
**第 11 週：神經網路基礎** -- 感知器、激活函數、前向/反向傳播、正則化與 Batch Normalization